# 05j-m — Train support rigenerativo e fresh test sigillato

05j-l ha dimostrato che un gate scelto sul vecchio fit non riconosce l'inversione di dominio del decoder. Questo notebook genera quindi **solo** nuovo supporto teacher destinato al training: 96 coppie causali near-regenerative, tutte conservate. Contemporaneamente congela il piano di 32 coppie fresh-test con seed separati, ma non ne genera gli outcome. Il vecchio 05j-i è da ora development, non test. Non viene addestrato alcun modello. Usare una sessione Kaggle nuova con persistenza `No persistence` o `Files only`, mai la persistenza delle variabili NEURON.

## 1. Checkout riproducibile e teacher canonico

In [ ]:
import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git'; ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main')
TEACHER_REPOSITORY='https://github.com/SelfishGene/neuron_as_deep_net.git'; TEACHER_COMMIT='074c4666300a8ad246601dab179a97a6942f0f29'
ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd().resolve(); WORKSPACE=ROOT/'hayflow_workspace'; WORKSPACE.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None): print('+',' '.join(map(str,command)),flush=True); subprocess.run(list(map(str,command)),cwd=cwd,check=True)
override=os.environ.get('HAYFLOW_ELM_REPO'); mounted=[Path(override).expanduser()] if override else []; mounted.extend([Path.cwd(),*Path.cwd().parents])
ELM_REPO=next((p.resolve() for p in mounted if (p/'src'/'hayflow_teacher').is_dir()),None)
if ELM_REPO is None:
    ELM_REPO=WORKSPACE/'elmneuron'
    if not (ELM_REPO/'.git').is_dir(): run(['git','clone',ELM_REPOSITORY,ELM_REPO])
    run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO); run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO)
TEACHER_REPO=Path(os.environ.get('HAYFLOW_TEACHER_REPO',ELM_REPO.parent/'neuron_as_deep_net')).expanduser().resolve()
if not (TEACHER_REPO/'.git').is_dir(): run(['git','clone',TEACHER_REPOSITORY,TEACHER_REPO])
run(['git','checkout','--detach',TEACHER_COMMIT],cwd=TEACHER_REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=TEACHER_REPO,text=True).strip()==TEACHER_COMMIT
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip(); print({'owned':str(ELM_REPO),'teacher':str(TEACHER_REPO),'revision':REVISION})

## 2. Dipendenze e MOD originali (CPU)

In [ ]:
run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7','numpy','pandas','matplotlib','h5py','pyarrow','pyyaml'])
SIMULATION_DIR=TEACHER_REPO/'L5PC_NEURON_simulation'
if not list(SIMULATION_DIR.rglob('libnrnmech.so')):
    nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl'); run([nrnivmodl,'mods'],cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob('libnrnmech.so'))
sys.path.insert(0,str(ELM_REPO))
for name in tuple(sys.modules):
    if name=='src.hayflow_teacher' or name.startswith('src.hayflow_teacher.') or name=='src.hayflow_data' or name.startswith('src.hayflow_data.'): sys.modules.pop(name,None)
importlib.invalidate_caches(); print('Teacher compilato; GPU non richiesta.')

## 3. Individuazione sicura degli input esatti

In [ ]:
INPUT_ROOT=Path('/kaggle/input')
def extract_zip_safely(source,destination):
    source,destination=Path(source),Path(destination); destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target=(destination/member.filename).resolve(); assert target.is_relative_to(destination.resolve()),member.filename
        archive.extractall(destination)
    return destination
def valid_base(path):
    path=Path(path); return (path/'transition_dataset.h5').is_file() and (path/'targeted_pilot'/'candidate_trials.parquet').is_file() and (path/'artifact_index.json').is_file()
base_candidates=([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else [])
if INPUT_ROOT.is_dir(): base_candidates += [p.parent.parent for p in INPUT_ROOT.rglob('candidate_trials.parquet') if p.parent.name=='targeted_pilot']
BASE_DATASET=next((p.resolve() for p in base_candidates if valid_base(p)),None)
if BASE_DATASET is None and INPUT_ROOT.is_dir():
    archives=[p for p in INPUT_ROOT.rglob('*.zip') if 'targeted-transition-dataset' in str(p).lower() or 'targeted_transition_dataset' in p.name.lower()]
    assert len(archives)==1,f'Archivio base ambiguo o assente: {archives}'
    extracted=extract_zip_safely(archives[0],'/kaggle/temp/hayflow05jm_base'); BASE_DATASET=next((p.parent.parent for p in extracted.rglob('candidate_trials.parquet') if valid_base(p.parent.parent)),None)
assert BASE_DATASET is not None,'Base v1.1 completa non trovata.'
calibration_candidates=([Path(os.environ['HAYFLOW_CALIBRATION_SOURCE']).expanduser()] if os.environ.get('HAYFLOW_CALIBRATION_SOURCE') else [])
calibration_candidates += [Path('/kaggle/input/datasets/alessandrobelli/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration'),Path('/kaggle/input/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration')]
if INPUT_ROOT.is_dir(): calibration_candidates += [p.parent for p in INPUT_ROOT.rglob('selected_dendritic_protocols.json')]
CALIBRATION_SOURCE=next((p.resolve() for p in calibration_candidates if (p/'selected_dendritic_protocols.json').is_file()),None)
assert CALIBRATION_SOURCE is not None,'Calibrazione 01b non trovata.'
from src.hayflow_teacher.regenerative_training_support import EXPECTED_05JI_INDEX_SHA256,EXPECTED_05JL_INDEX_SHA256
import hashlib
def exact_artifact(path,index_hash,zip_name):
    path=Path(path)
    if path.is_file(): return path.name==zip_name
    return any(hashlib.sha256(idx.read_bytes()).hexdigest()==index_hash for idx in path.rglob('artifact_index.json'))
def locate(env_name,index_hash,zip_name):
    candidates=[Path(os.environ[env_name]).expanduser()] if os.environ.get(env_name) else []
    if INPUT_ROOT.is_dir(): candidates += list(INPUT_ROOT.rglob(zip_name))+[p.parent for p in INPUT_ROOT.rglob('artifact_index.json')]
    return next((p.resolve() for p in candidates if p.exists() and exact_artifact(p,index_hash,zip_name)),None)
ARTIFACT_05JI_SOURCE=locate('HAYFLOW_05JI_ARTIFACT',EXPECTED_05JI_INDEX_SHA256,'hayflow_regenerative_confirmation_support.zip')
ARTIFACT_05JL_SOURCE=locate('HAYFLOW_05JL_ARTIFACT',EXPECTED_05JL_INDEX_SHA256,'hayflow_hines_residual_safety_gate.zip')
assert ARTIFACT_05JI_SOURCE is not None,'Artefatto esatto 05j-i non trovato.'
assert ARTIFACT_05JL_SOURCE is not None,'Artefatto esatto 05j-l non trovato.'
print({'base':str(BASE_DATASET),'calibration':str(CALIBRATION_SOURCE),'05j-i':str(ARTIFACT_05JI_SOURCE),'05j-l':str(ARTIFACT_05JL_SOURCE)})

## 4. Teacher, base immutabile e provenienza

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_teacher import RegenerativeTrainingSupportConfig,RegenerativeTrainingSupportSession,expected_audit_hashes
TARGET_CONFIG_PATH=ELM_REPO/'configs'/'hayflow'/'targeted_transition_dataset_v1_1.yml'
CONFIG_PATH=ELM_REPO/'configs'/'hayflow'/'hayflow_regenerative_training_support.yml'
payload=yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')); acquisition_config=RegenerativeTrainingSupportConfig.from_mapping(payload['regenerative_training_support'])
target_config=yaml.safe_load(TARGET_CONFIG_PATH.read_text(encoding='utf-8'))
OUTPUT_DIR=Path(os.environ.get('HAYFLOW_OUTPUT_DIR',ROOT/'artifacts'/'hayflow_regenerative_training_support')).expanduser().resolve()
if OUTPUT_DIR.exists(): raise RuntimeError(f'Output protetto già presente: {OUTPUT_DIR}. Usa una nuova HAYFLOW_OUTPUT_DIR o una sessione nuova.')
session=RegenerativeTrainingSupportSession(ELM_REPO,TEACHER_REPO,base_dataset=BASE_DATASET,calibration_source=CALIBRATION_SOURCE,dataset_config_path=TARGET_CONFIG_PATH,output_dir=OUTPUT_DIR,seed=314159,expected_teacher_hashes=expected_audit_hashes(),native_snapshot_stride=target_config['storage']['native_snapshot_stride_ms'],artifact_05ji_source=ARTIFACT_05JI_SOURCE,artifact_05jl_source=ARTIFACT_05JL_SOURCE,acquisition_config=acquisition_config)
teacher_report=session.prepare_teacher(); contract_report=session.prepare_targeted_contract(); base_report=session.verify_base_dataset(verify_large_hdf=True); equilibrium_report=session.import_base_equilibrium(); provenance_report=session.verify_prerequisites()
display({'teacher':teacher_report,'contract':contract_report,'base':base_report,'equilibrium':equilibrium_report,'provenance':provenance_report}); assert teacher_report['segment_count']==642 and base_report['valid'] and provenance_report['valid']

## 5. Congelamento simultaneo dei piani train e fresh-test
I due namespace di seed vengono fissati prima di osservare qualsiasi nuovo outcome. Lo snapshot bank viene costruito soltanto per il train: il fresh test resta non simulato.

In [ ]:
train_protocols,acquisition_contract=session.build_acquisition_plans()
snapshot_report=session.prepare_snapshot_bank(train_protocols,conditioning_ms=acquisition_config.conditioning_ms)
display({'train':{k:v for k,v in acquisition_contract['train'].items() if k!='pairs'},'fresh_test':{k:v for k,v in acquisition_contract['fresh_test'].items() if k not in {'pairs','protocols'}},'snapshots':snapshot_report}); assert acquisition_contract['seed_overlap']==[] and acquisition_contract['fresh_test']['outcomes_generated'] is False and snapshot_report['snapshot_count']==acquisition_config.train_pair_count

## 6. Generazione del solo shard train
Vengono generate 192 traiettorie (96 coppie) e 2.304 transizioni. Il fresh test non viene eseguito.

In [ ]:
manifest=session.generate_training_shard(train_protocols)
display({'dataset_role':manifest['dataset_role'],'trajectory_count':manifest['trajectory_count'],'transition_count':manifest['transition_count']}); assert manifest['trajectory_count']==192 and manifest['transition_count']==2304

## 7. Replay esaustivo, supporto realizzato e isolamento del test
`valid=True` certifica integrità, replay e assenza dei seed fresh-test. `scientific_train_support_sufficient` indica separatamente se almeno 72 coppie sono realmente near-regenerative. Nessun modello viene autorizzato.

In [ ]:
final_report=session.validate_training_shard(train_protocols)
display({'valid':final_report['valid'],'support_sufficient':final_report['scientific_train_support_sufficient'],'diagnosis':final_report['diagnosis'],'strata':final_report['realized_train_stratum_counts'],'fresh_test':final_report['fresh_test'],'replay':final_report['exhaustive_replay'],'next_step':final_report['next_step']}); assert final_report['valid'] and final_report['fresh_test']['outcomes_generated'] is False and not final_report['candidate_authorized']

## 8. Scarica lo ZIP con il metodo browser Blob

In [ ]:
from pathlib import Path
import base64, shutil
from IPython.display import Javascript, display
zip_base=Path('/kaggle/working/hayflow_regenerative_training_support')
zip_path=Path(shutil.make_archive(str(zip_base),'zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name))
encoded=base64.b64encode(zip_path.read_bytes()).decode('ascii'); filename=zip_path.name
display(Javascript(f"""
const b64 = '{encoded}';
const bytes = new Uint8Array(atob(b64).split('').map(c => c.charCodeAt(0)));
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const a = document.createElement('a'); a.href = url; a.download = '{filename}';
document.body.appendChild(a); a.click(); a.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
""")); print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,1)})